# Filamentation 1030 nm / 13 µJ -- fluence, I_max, rho_e, rho_s, cycles, L_c

Solveur : `NewSim3juillet.py` (uploadé, copié dans `sim_1030nm_experiment/`).
Distinct du paquet modulaire `sim/filament_sim.py` utilisé par
`term_ablation_study.ipynb` : interrupteurs plus grossiers
(`enable_kerr`/`enable_avalanche`/`enable_recombination` seulement), et
**pas de masque spectral** (voir §2).

Géométrie et énergie reprises de `notebooksimu3juillet.ipynb` (cellule 2),
recoupées contre `unified_filament_slider_v3.py` : les deux calculent
indépendamment `Z_FOCUS_GLASS_DIST_UM = 1.45 × 272 = 394.4 µm` (le premier
l'arrondit à 394.0) -- accord à 0.1 %, donc un vrai recoupement.

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.special import jn_zeros
from scipy.constants import c as c_SI, epsilon_0, m_e, elementary_charge as q_e

sys.path.insert(0, str(Path.cwd().parent / "sim_1030nm_experiment"))
sys.path.insert(0, str(Path.cwd() / "sim_1030nm_experiment"))

from NewSim3juillet import run, n_sellmeier

OUT_ROOT = Path("runs_1030nm"); OUT_ROOT.mkdir(exist_ok=True)
FIG_DIR = OUT_ROOT / "figures_for_report"; FIG_DIR.mkdir(exist_ok=True)
print("Imports OK")

## 1. Paramètres d'entrée du calcul

### Géométrie
Foyer géométrique à `Z_FOCUS_AIR_DIST_UM = 272 µm` *dans l'air* ; réfracté par
l'interface (`N_GLASS = 1.45`) il recule à `394.4 µm` sous la surface. Le
solveur est centré sur ce foyer (`z_sim = 0`), donc la face d'entrée est à
`begin = -394.4 µm`.

### Énergie -- deux corrections avant `run()`
1. **Fresnel** en incidence normale : `T = 1-((n0-1)/(n0+1))² = 96.63 %`
   → 13.0 µJ incidents deviennent **12.56 µJ dans le verre**.
2. **Convention flat-top de `Config.__post_init__`** : la formule
   `I0 = 2·energy_uJ/(π w0² Δt)` traite `Δt` comme une durée flat-top alors que
   le profil est gaussien. Vérifié analytiquement : pour un champ
   `exp(-t²/tp²)` avec `tp = Δt/√(2 ln2)`, l'énergie vaut
   `I0·(πw0²/2)·(tp√(π/2))` et `tp√(π/2) = 1.0645·Δt`. Il faut donc diviser
   par `√(π/(4 ln2)) = 1.0645` → `energy_uJ = 11.80` envoyé à `run()`.
   `[init] U_beam(0)` doit alors afficher ≈ **12.56 µJ**, pas 11.80 -- c'est
   le contrôle que la correction est bien appliquée.

### n2 -- correction de ma version précédente de ce notebook
J'avais écrit que `notebooksimu3juillet.ipynb` utilisait le défaut de `Config`
(`2.4e-20`). **C'est faux** : `run()` a sa propre valeur par défaut
`n2=3.54e-20` (ligne 611) qu'elle passe explicitement à `Config`, donc le
défaut de `Config` n'est jamais atteint via `run()`. Le run original utilisait
donc **3.54e-20 m²/W** = la valeur de Couairon 2005 **à 800 nm**.

La question de fond reste : cette expérience est à **1030 nm**. Les trois
valeurs candidates sont gardées en parallèle ci-dessous plutôt que tranchées
en silence.

| n2 (m²/W) | source | P_cr | P_in/P_cr |
|---|---|---|---|
| 2.40e-20 | Milam 1998, 800 nm (Taylor) | 4.57 MW | 9.8 |
| 2.74e-20 | Milam 1998, **1053 nm** (le plus proche de 1030) | 4.01 MW | 11.2 |
| 3.54e-20 | Couairon 2005, 800 nm (**défaut de `run()`**) | 3.10 MW | 14.5 |

La ligne 2.74e-20 est celle déjà utilisée §8 du rapport pour annoncer
`P_cr ≈ 4.0 MW` à 1030 nm -- cohérence retrouvée à 0.1 %.

In [ ]:
# ============ PARAMETRES D'ENTREE DU CALCUL ============

# --- Geometrie ---
Z_FOCUS_AIR_DIST_UM   = 272.0
N_GLASS               = 1.45
Z_FOCUS_GLASS_DIST_UM = N_GLASS * Z_FOCUS_AIR_DIST_UM     # 394.4 um
BEGIN_M = -Z_FOCUS_GLASS_DIST_UM * 1e-6                    # face d'entree
END_M   =  800e-6                                          # 800 um apres le foyer

# --- Laser ---
WAVELENGTH_M    = 1030e-9
ENERGY_INPUT_UJ = 13.0        # AVANT l'interface
W0_M            = 3e-6        # waist AU FOYER (pas a l'entree ! cf. section 3)
DELTA_T_S       = 263e-15     # FWHM en intensite

N0_PUMP       = 1.45
TRANSMISSION  = 1.0 - ((N0_PUMP - 1.0) / (N0_PUMP + 1.0))**2
ENERGY_IN_GLASS = ENERGY_INPUT_UJ * TRANSMISSION            # 12.561 uJ
GAUSS_FLATTOP   = float(np.sqrt(np.pi / (4.0 * np.log(2)))) # 1.0645
ENERGY_SIM_UJ   = ENERGY_IN_GLASS / GAUSS_FLATTOP           # 11.801 uJ -> run()

# --- Materiau (SiO2) ---
N2_MILAM_800   = 2.40e-20   # Milam 1998 @800nm  (defaut de Config, jamais atteint via run())
N2_MILAM_1053  = 2.74e-20   # Milam 1998 @1053nm -- le plus proche de 1030 nm
N2_COUAIRON    = 3.54e-20   # Couairon 2005 @800nm -- DEFAUT DE run(), utilise par le run original
N2_CHOSEN      = N2_MILAM_1053

UI_EV       = 9.0        # gap SiO2 -- Couairon 2005, Bulgakova 2010
MEFF_REL    = 0.64       # masse reduite -- Couairon 2005
TAU_C_S     = 1.7e-15    # defaut Config : omega0*tau_c = 3.1 a 1030 nm (convention Bulgakova,
                          # PAS Couairon qui utilise 1e-14 -> omega0*tau_c = 23.6)
TAU_R_S     = 330e-15    # piegeage STE -- Mouskeftaras 2013 / Tsaturyan 2025 (Couairon: 150 fs)
RHO_MAX_CM3 = 2.1e22     # densite d'atomes -- Couairon 2005 (Bulgakova: 6.6e22)
US_EV       = 6.0        # niveau STE -- Chimier PRB 2011 (defaut Config)
F_R, TAU_D_S, TAU_S_S = 0.18, 32e-15, 12e-15   # Raman -- Couairon 2005

# --- Sonde ---
LAMBDA_PROBE_M = 490e-9

# --- Grille (voir section 2 pour la justification / les reserves) ---
NZ_TARGET_DZ_M = 24e-9
LZ = END_M - BEGIN_M
NZ = int(LZ / NZ_TARGET_DZ_M)
NT = 4096      # RELEVE de 2000 -> 4096, voir section 2
NR = 3001      # ordre Hankel N -> N-1 = 3000 points radiaux
R_FACTOR = 90.0
SAVE_STRIDE  = 100
RHO_T_STRIDE = 20   # RELEVE de 10 -> 20 pour diviser la taille du npz par 2

n0_1030 = n_sellmeier(WAVELENGTH_M)
k0 = 2*np.pi*n0_1030/WAVELENGTH_M
zR_focal = k0*W0_M**2/2
W_ENTRANCE_M = W0_M*np.sqrt(1+(BEGIN_M/zR_focal)**2)   # rayon REEL a l'entree

print(f"n0(1030nm)          = {n0_1030:.4f}   (vs N0_PUMP={N0_PUMP} suppose ailleurs -> accord)")
print(f"Foyer sous surface  = {Z_FOCUS_GLASS_DIST_UM:.1f} um")
print(f"Boite z_sim         = [{BEGIN_M*1e6:+.1f}, {END_M*1e6:+.1f}] um")
print(f"Energie   13.0 uJ -> {ENERGY_IN_GLASS:.3f} uJ (Fresnel) -> energy_uJ={ENERGY_SIM_UJ:.3f} envoye a run()")
print(f"z_R au foyer        = {zR_focal*1e6:.2f} um")
print(f"RAYON A L'ENTREE    = {W_ENTRANCE_M*1e6:.2f} um   <-- PAS 3 um, cf. section 3")

## 2. Vérification des paramètres de grille

### Longitudinal -- OK (voire large)
`dz = 24 nm`, `Nz = 49766`. Phase Kerr par pas à l'intensité de clampage :
`k0·n2·I·dz = 3.8e-3 rad`, très en dessous du critère usuel 0.05. Un cœur de
filament de 1 µm a `z_R = 4.4 µm`, soit 184 pas par longueur de Rayleigh.
**On pourrait passer à `dz = 48 nm` et diviser le coût par 2** sans rien perdre.

### Radial -- OK
`R_max = 90 × 3 µm = 270 µm`, absorbeur à 243 µm. La grille de Hankel est
quasi-uniforme (`dr = 89.2 → 90.0 nm`), soit **33 points dans w0** et
**11 points dans un cœur de 1 µm** (acceptable, pas généreux). Marges par
rapport au faisceau linéaire : **×8.1 à l'entrée** (29.9 µm), **×4.0 en sortie**
(60.4 µm à z=+800 µm). Matrice de Hankel `Y` : 3000×3000 float64 = 72 MB, sans
problème.

### Temporel -- LE POINT FAIBLE
Avec `Nt = 2000` (valeur de `notebooksimu3juillet.ipynb`) :
`tp = 223 fs`, `tmax = 5tp = 1117 fs`, `dt = 1.117 fs`.

- Fenêtre spectrale en fréquence **absolue** : de **−0.157 à +0.738 PHz**, soit
  seulement **406 nm** au bord bleu. Le rapport `f_Nyquist/f0 = 1.54`, contre
  **2.01** pour la run 800 nm déjà validée du solveur modulaire → cette run est
  *relativement plus serrée* que celle qui a servi de référence.
- L'ordre multiphotonique passe de **K = 6** (800 nm) à **K = 8** (1030 nm,
  E_photon = 1.204 eV) : le taux d'ionisation est encore plus sensible à
  l'intensité, ce qui plaide pour un échantillonnage temporel *plus* fin, pas
  moins.

→ `Nt` relevé à **4096** ci-dessus (bord bleu à 248 nm, `f_Nyq/f0 = 3.15`).

### Bug potentiel : `T_op` non masqué
`NewSim3juillet.py` ligne 299 : `T_op = 1.0 + ff / cfg.frequency`, **sans
masque spectral**. `T_op < 0` correspond à une fréquence absolue négative
(non physique) : **17.5 % des bins à Nt=2000, 34 % à Nt=4096**. Le bruit
numérique qui atterrit là est amplifié avec le mauvais signe.

Ce n'est pas une critique gratuite : le **même solveur borne déjà la
dispersion** (ligne 283, `omega_safe = np.clip(..., 2πc/5µm, 2πc/0.18µm)`) mais
pas le self-steepening -- c'est une incohérence interne. Le paquet modulaire
`sim/grids.py` corrige exactement ça avec un `spec_mask` en tanh, dont le
commentaire dit mot pour mot *« so the region where it would be negative
(unphysical, omega < 0) can never amplify anything »*.

Vérifié numériquement : avec ce masque, `min(T_op)` passe de **−0.54 à −0.0000**
(Nt=2000) et de **−2.15 à −0.0000** (Nt=4096). Patch de ~6 lignes proposé en
section 2bis.

### Sauvegarde -- c'est ça, le npz « trop lourd »
`save_stride=100` → 498 plans, `dz_save = 2.40 µm`. Suffisant pour un
`L_c,f ≈ 220 µm` ; **marginal** pour résoudre des pincements individuels de
quelques µm.

`rho_t_stride=10` → 3 cubes (z,r,t) de **1.20 GB chacun = 3.6 GB**, contre
18 MB seulement pour tous les tableaux 2D. **C'est l'unique cause du fichier
trop lourd.** Options : `rho_t_stride=20` → 1.8 GB (retenu ici),
`40` → 0.9 GB, `0` → 18 MB (mais on perd le slider Abel).

In [ ]:
# --- Verification automatique de la grille (a relancer si on change les params) ---
dz = LZ/NZ
print("=== LONGITUDINAL ===")
print(f"dz={dz*1e9:.2f} nm  Nz={NZ}")
print(f"  phase Kerr/pas @I_clamp: {k0*N2_CHOSEN*5e17*dz:.2e} rad  (critere <0.05)")
print(f"  z_R(coeur 1um)={k0*(1e-6)**2/2*1e6:.2f} um -> {k0*(1e-6)**2/2/dz:.0f} pas/z_R")

print("\n=== RADIAL ===")
j = jn_zeros(0, NR); R = R_FACTOR*W0_M
rlist = j[:NR-1]*R/j[NR-1]; dr = np.diff(rlist)
print(f"R_max={R*1e6:.1f} um  absorbeur={0.9*R*1e6:.1f} um  Nr={NR-1}")
print(f"  dr={dr.min()*1e6:.4f}..{dr.max()*1e6:.4f} um -> {W0_M/dr.mean():.1f} pts dans w0, {1e-6/dr.mean():.1f} dans 1um")
w_end = W0_M*np.sqrt(1+(END_M/zR_focal)**2)
print(f"  marge vs faisceau lineaire: entree x{0.9*R/W_ENTRANCE_M:.1f}, sortie x{0.9*R/w_end:.1f}")
print(f"  matrice Hankel Y: {(NR-1)**2*8/1e6:.0f} MB")

print("\n=== TEMPOREL ===")
tp = DELTA_T_S/np.sqrt(2*np.log(2)); tmax = 5*tp; dt = 2*tmax/NT
f0 = c_SI/WAVELENGTH_M; ff = np.fft.fftfreq(NT, d=dt); fabs = f0+ff
print(f"tp={tp*1e15:.1f} fs  tmax={tmax*1e15:.0f} fs  dt={dt*1e15:.3f} fs  Nt={NT}")
print(f"  f_Nyq/f0={1/(2*dt)/f0:.2f}  (run 800nm validee: 2.01)")
print(f"  fenetre absolue: {fabs.min()/1e15:+.3f}..{fabs.max()/1e15:+.3f} PHz  (bord bleu {c_SI/fabs.max()*1e9:.0f} nm)")
Top = 1+ff/f0
print(f"  T_op non masque: min={Top.min():+.3f}  ({np.mean(Top<0)*100:.1f}% des bins NEGATIFS)")
print(f"  K multiphotonique = {int(np.ceil(UI_EV/(1240/(WAVELENGTH_M*1e9))))} photons "
      f"(E_ph={1240/(WAVELENGTH_M*1e9):.3f} eV)")

print("\n=== SAUVEGARDE ===")
n_saves = NZ//SAVE_STRIDE+1; Nt_sub = (NT-1)//RHO_T_STRIDE+1
cube = n_saves*(NR-1)*Nt_sub*4
print(f"n_saves={n_saves}  dz_save={dz*SAVE_STRIDE*1e6:.2f} um")
print(f"Nt_sub={Nt_sub}  dt_sub={dt*RHO_T_STRIDE*1e15:.1f} fs")
print(f"3 cubes (z,r,t) = {3*cube/1e9:.2f} GB   |   tableaux 2D = {3*n_saves*(NR-1)*4/1e6:.0f} MB")

## 2bis. Patch proposé pour `T_op` (masque spectral)

À appliquer dans `sim_1030nm_experiment/NewSim3juillet.py`, en remplacement de
la ligne 299. Porté tel quel de `sim/grids.py`. **Non appliqué automatiquement**
-- c'est une modification du solveur, à toi de décider.

```python
# AVANT (ligne 299) :
# T_op = 1.0 + ff / cfg.frequency

# APRES :
u_norm = (cfg.frequency + ff) / cfg.frequency
u_lo = (c / 5.0e-6)  / cfg.frequency      # ~0.206 : lambda = 5 um
u_hi = (c / 0.18e-6) / cfg.frequency      # ~5.722 : lambda = 180 nm
w_edge = 0.05
spec_mask = 0.25 * (1.0 + cp.tanh((u_norm - u_lo) / w_edge)) \
                 * (1.0 + cp.tanh((u_hi - u_norm) / w_edge))
T_op = (1.0 + ff / cfg.frequency) * spec_mask
```

Effet vérifié : `min(T_op)` passe de −2.15 à −0.0000 à Nt=4096, en supprimant
37 % des bins -- tous situés hors de la fenêtre de validité de Sellmeier
[180 nm, 5 µm], donc sans contenu physique à perdre.

## 3. P_cr et L_c -- correction importante sur le rayon d'entrée

**Erreur dans ma version précédente** : j'avais mis `w0 = 3 µm` (le waist *au
foyer*) dans la formule de Marburger. Or `L_DF = k0 w0²/2` doit être la
longueur de Rayleigh du faisceau **au plan d'entrée**, là où il est lancé.

Le solveur lance `envelope_gaussian_focused` avec
`curv = 1 + 2i·begin/b`, `b = k0 w0²` : à `z = begin = −394.4 µm`, le rayon
réel est `w_in = w0·|curv| = 29.9 µm`, soit **10× le waist focal**.

Conséquence, avec `n2 = 3.54e-20` (ce qu'utilisait le run original) :

| | `w0 = 3 µm` (faux) | `w_in = 29.9 µm` (correct) |
|---|---|---|
| `L_DF` | 39.8 µm | 3948 µm |
| `L_c` | 5.0 µm | 491 µm |
| `L_c,f` | 4.9 µm | **219 µm** |
| collapse à `z_sim` | −389 µm (collé à l'entrée) | **−176 µm** |

La valeur corrigée est physiquement sensée : le foyer non-linéaire arrive
**176 µm avant le foyer géométrique**, ce qui est le résultat classique
(le Kerr fait collapser en amont de la lentille). La valeur fausse plaçait le
collapse à 5 µm de la face d'entrée, ce qui n'avait aucun sens pour un
faisceau qui y fait encore 30 µm de rayon.

In [ ]:
def marburger(n2, w_input, wavelength, n0, E_uJ_reel, delta_t, f_ext):
    """P_cr, P_in, L_c, L_c,f. w_input = rayon AU PLAN D'ENTREE (pas au foyer).
    E_uJ_reel = energie physique dans le verre (pas la valeur reduite -> run())."""
    P_cr = 3.77*wavelength**2/(8*np.pi*n0*n2)
    P0 = (E_uJ_reel*1e-6)/(GAUSS_FLATTOP*delta_t)
    ratio = P0/P_cr
    kk = 2*np.pi*n0/wavelength
    L_DF = kk*w_input**2/2
    inner = (np.sqrt(ratio)-0.852)**2 - 0.0219
    if ratio <= 0.852**2 or inner <= 0:
        return P_cr, P0, ratio, L_DF, np.nan, np.nan
    L_c = 0.367*L_DF/np.sqrt(inner)
    return P_cr, P0, ratio, L_DF, L_c, 1.0/(1.0/L_c + 1.0/f_ext)

F_EXT = abs(BEGIN_M)
PREDICTIONS = {}
for lab, n2 in (("Milam 800nm  2.40e-20", N2_MILAM_800),
                ("Milam 1053nm 2.74e-20", N2_MILAM_1053),
                ("Couairon    3.54e-20", N2_COUAIRON)):
    P_cr,P0,ratio,L_DF,L_c,L_cf = marburger(
        n2, W_ENTRANCE_M, WAVELENGTH_M, n0_1030, ENERGY_IN_GLASS, DELTA_T_S, F_EXT)
    z_pred = L_cf*1e6 + BEGIN_M*1e6
    PREDICTIONS[lab] = dict(n2=n2, P_cr=P_cr, ratio=ratio, L_c=L_c, L_cf=L_cf, z_pred_um=z_pred)
    print(f"{lab}: P_cr={P_cr*1e-6:5.2f} MW  P_in={P0*1e-6:.2f} MW  P_in/P_cr={ratio:5.2f}")
    print(f"{'':22s}  L_DF={L_DF*1e6:7.0f} um  L_c={L_c*1e6:6.1f} um  L_c,f={L_cf*1e6:6.1f} um"
          f"  -> collapse a z_sim={z_pred:+7.1f} um")
print(f"\n(foyer geometrique a z_sim=0 ; face d'entree a z_sim={BEGIN_M*1e6:+.1f} um)")

## 4. Lancer (ou recharger) la simulation

**Non exécuté ici** (pas de GPU dans cet environnement). Coût attendu :
`Nz≈49800` pas, chacun avec 4 produits Hankel denses 3000×3000 sur un champ
3000×4096 -- compter **plusieurs heures de GPU**.

Note connue : `NewSim3juillet.py::_record()` initialise `E_plasma_z`,
`E_MPI_z`, `E_STE_z` (lignes 484-486) mais ne leur assigne **jamais** de valeur
(vérifié : seules occurrences ultérieures = lectures lignes 582/591-593).
Contrairement à `sim/filament_sim.py`, **aucune figure de pertes d'énergie
n'est possible depuis ce solveur** tant que ce n'est pas corrigé.

In [ ]:
RUN_TAG = {N2_MILAM_800:"n2_800nm", N2_MILAM_1053:"n2_1053nm", N2_COUAIRON:"n2_couairon"}[N2_CHOSEN]
OUT_DIR = str(OUT_ROOT / f"filament_13uJ_w3um_{RUN_TAG}")
NPZ_PATH = Path(OUT_DIR)/"result.npz"

if NPZ_PATH.exists():
    print(f"Chargement : {NPZ_PATH}")
    res = dict(np.load(NPZ_PATH, allow_pickle=True))
else:
    print(f"Lancement -> {OUT_DIR}   (plusieurs heures de GPU)")
    res = run(
        Nz=NZ, Nt=NT, Nr=NR,
        begin=BEGIN_M, end=END_M, R_factor=R_FACTOR,
        wavelength=WAVELENGTH_M,
        energy_uJ=ENERGY_SIM_UJ,
        w0=W0_M, delta_t=DELTA_T_S,
        n2=N2_CHOSEN, Ui_eV=UI_EV,
        meff_rel=MEFF_REL, tau_c=TAU_C_S, tau_r=TAU_R_S,
        rho_max=RHO_MAX_CM3, Us_eV=US_EV,
        f_R=F_R, tau_d=TAU_D_S, tau_s=TAU_S_S,
        enable_ste=True,
        lambda_probe=LAMBDA_PROBE_M,
        rho_t_stride=RHO_T_STRIDE, save_stride=SAVE_STRIDE,
        out_dir=OUT_DIR, envelope="gaussian_focused",
    )
    print(f"Termine -> {OUT_DIR}/result.npz")

# CONTROLE : U_beam(0) affiche par le solveur doit valoir ~12.56 uJ (pas 11.80)
print(f"\nAttendu au demarrage : [init] U_beam(0) = {ENERGY_IN_GLASS:.3f} uJ")

In [ ]:
z_um = np.asarray(res["z"])*1e6            # 0 = foyer geometrique
r_um = np.asarray(res["r"])*1e6            # deja miroir +/-
i_axis = int(np.argmin(np.abs(r_um)))
Imax_z = np.asarray(res["Imax_z"])
rho_rz = np.asarray(res["rho_rz"])
rho_s_rz = np.asarray(res["rho_s_rz"])
fluence_rz = np.asarray(res["fluence_rz"])
I_CLAMP = 5e13

print(f"z : [{z_um[0]:+.0f}, {z_um[-1]:+.0f}] um ({len(z_um)} plans)")
print(f"r : [{r_um[0]:.1f}, {r_um[-1]:.1f}] um")
print(f"I_max     = {Imax_z.max():.3e} W/cm2 @ z={z_um[np.argmax(Imax_z)]:+.1f} um")
print(f"rho_e max = {rho_rz[:,i_axis].max():.3e} cm-3")
print(f"rho_s max = {rho_s_rz[:,i_axis].max():.3e} cm-3")

## 5. Fluence -- cycles de focalisation / défocalisation

In [ ]:
fig, ax = plt.subplots(figsize=(11,4))
cs = ax.contour(z_um, r_um, fluence_rz.T, levels=(1.,2.,5.,10.,20.), colors="black", linewidths=0.8)
ax.clabel(cs, inline=True, fontsize=6, fmt="%.0f J/cm2")
for lab, p in PREDICTIONS.items():
    ax.axvline(p["z_pred_um"], ls="--", lw=1, label=f"L_c,f {lab.split()[0]}")
ax.axvline(0.0, color="purple", ls=":", lw=1.2, label="foyer geometrique")
ax.axvline(BEGIN_M*1e6, color="green", ls=":", lw=1.2, label="face d'entree")
ax.set_xlabel("z (um, 0 = foyer geometrique)"); ax.set_ylabel("r (um)")
ax.set_title("Contours de fluence -- cycles focalisation/defocalisation")
ax.legend(fontsize=7, ncol=2); fig.tight_layout()
fig.savefig(FIG_DIR/"fluence_contours_1030nm.png", dpi=150)

## 6. Intensité crête vs z -- L_c,f prédite vs collapse observé

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(z_um, Imax_z, lw=1.6, color="black", label="I_max(z)")
ax.axhline(I_CLAMP, ls="--", color="crimson", lw=1, label=f"I_clamp~{I_CLAMP:.0e}")
for (lab,p),col in zip(PREDICTIONS.items(), ("tab:blue","tab:green","tab:orange")):
    ax.axvline(p["z_pred_um"], ls="--", lw=1.2, color=col,
               label=f"L_c,f {lab.split()[0]} (P/Pcr={p['ratio']:.1f})")
ax.axvline(0.0, color="purple", ls=":", lw=1.2, label="foyer geometrique")
ax.set_yscale("log"); ax.set_xlabel("z (um, 0 = foyer geometrique)")
ax.set_ylabel("Peak intensity (W/cm2)")
ax.set_title("Intensite crete -- L_c,f predite vs collapse observe")
ax.legend(fontsize=7); fig.tight_layout()
fig.savefig(FIG_DIR/"peak_intensity_vs_Lc_1030nm.png", dpi=150)

## 7. Électrons libres (rho_e) vs excitons auto-piégés (rho_s)

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(z_um, np.clip(rho_rz[:,i_axis],1e-3,None), lw=1.6, color="black", label="rho_e (libres)")
ax.plot(z_um, np.clip(rho_s_rz[:,i_axis],1e-3,None), lw=1.6, color="tab:blue", label="rho_s (STE)")
ax.axhline(RHO_MAX_CM3, ls=":", color="gray", lw=1, label=f"rho_max={RHO_MAX_CM3:.1e}")
nc = epsilon_0*m_e*(2*np.pi*c_SI/LAMBDA_PROBE_M)**2/q_e**2*1e-6
ax.axhline(nc, ls="-.", color="purple", lw=1, label=f"n_c(sonde 490nm)={nc:.2e}")
ax.set_yscale("log"); ax.set_xlabel("z (um, 0 = foyer geometrique)")
ax.set_ylabel("On-axis rho (cm-3)")
ax.set_title("Electrons libres vs excitons auto-pieges, on-axis")
ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(FIG_DIR/"rho_e_rho_s_vs_z_1030nm.png", dpi=150)

## 8. Comptage des cycles + confrontation à L_c,f

In [ ]:
peaks_idx,_ = find_peaks(Imax_z, height=I_CLAMP)
print(f"{len(peaks_idx)} maximum(aux) local(aux) au-dessus de I_clamp :")
for i,ip in enumerate(peaks_idx):
    print(f"  cycle {i+1}: z={z_um[ip]:+8.1f} um   I={Imax_z[ip]:.3e} W/cm2")
if len(peaks_idx)>=2:
    print("Espacements (um):", np.round(np.diff(z_um[peaks_idx]),1))

if len(peaks_idx)>=1:
    z_obs = z_um[peaks_idx[0]]
    print(f"\nPremier collapse OBSERVE a z_sim = {z_obs:+.1f} um")
    for lab,p in PREDICTIONS.items():
        print(f"  vs {lab}: predit {p['z_pred_um']:+7.1f} um   ecart {z_obs-p['z_pred_um']:+7.1f} um")

## 9. Récapitulatif pour `main.tex`

In [ ]:
print("Figures :", FIG_DIR.resolve())
for f in sorted(FIG_DIR.glob("*.png")): print(" -", f.name)
print(f"\n1030 nm | {ENERGY_INPUT_UJ} uJ incident -> {ENERGY_IN_GLASS:.2f} uJ dans le verre")
print(f"w0(foyer)={W0_M*1e6:.1f} um, w(entree)={W_ENTRANCE_M*1e6:.1f} um, FWHM={DELTA_T_S*1e15:.0f} fs")
print(f"n2={N2_CHOSEN:.2e} m2/W ({RUN_TAG})  P_in/P_cr={PREDICTIONS['Milam 1053nm 2.74e-20']['ratio']:.1f}")
print(f"I_max={Imax_z.max():.3e} W/cm2 @ z={z_um[np.argmax(Imax_z)]:+.1f} um")
print(f"{len(peaks_idx)} cycle(s) detecte(s)")